# nuScenes devkit tutorial

Welcome to the nuScenes tutorial. This demo assumes the database itself is available at `/data/sets/nuscenes`, and loads a mini version of the full dataset.

## A Gentle Introduction to nuScenes

In this part of the tutorial, let us go through a top-down introduction of our database. Our dataset comprises of elemental building blocks that are the following:

1. `log` - Log information from which the data was extracted.
2. `scene` - 20 second snippet of a car's journey.
3. `sample` - An annotated snapshot of a scene at a particular timestamp.
4. `sample_data` - Data collected from a particular sensor.
5. `ego_pose` - Ego vehicle poses at a particular timestamp.
6. `sensor` - A specific sensor type.
7. `calibrated sensor` - Definition of a particular sensor as calibrated on a particular vehicle.
8. `instance` - Enumeration of all object instance we observed.
9. `category` - Taxonomy of object categories (e.g. vehicle, human).
10. `attribute` - Property of an instance that can change while the category remains the same.
11. `visibility` - Fraction of pixels visible in all the images collected from 6 different cameras.
12. `sample_annotation` - An annotated instance of an object within our interest.
13. `map` - Map data that is stored as binary semantic masks from a top-down view.

The database schema is visualized below. For more information see the [nuScenes schema](https://github.com/nutonomy/nuscenes-devkit/blob/master/docs/schema_nuscenes.md) page.
![](https://www.nuscenes.org/public/images/nuscenes-schema.svg)

## Google Colab (optional)

<br>
<a href="https://colab.research.google.com/github/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" align="left">
</a>
<br>

If you are running this notebook in Google Colab, you can uncomment the cell below and run it; everything will be set up nicely for you. Otherwise, manually set up everything.

In [ ]:
NUSCENES_DIR = "data/sets/nuscenes"

In [ ]:
!mkdir -p data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

!wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

!tar -xf v1.0-mini.tgz -C data/sets/nuscenes  # Uncompress the nuScenes mini split.

!pip install nuscenes-devkit &> /dev/null  # Install nuScenes.

--2026-09-12 03:29:35--  https://www.nuscenes.org/data/v1.0-mini.tgz
Resolving www.nuscenes.org (www.nuscenes.org)... 13.224.30.62, 13.224.30.18, 13.224.30.81, ...
Connecting to www.nuscenes.org (www.nuscenes.org)|13.224.30.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4167696325 (3.9G) [application/x-tar]
Saving to: ‘v1.0-mini.tgz’

v1.0-mini.tgz       100%[===================>]   3.88G  60.4MB/s    in 66s     

2026-09-12 03:30:42 (60.3 MB/s) - ‘v1.0-mini.tgz’ saved [4167696325/4167696325]



## Initialization

In [2]:
get_ipython().system('pip install --force-reinstall numpy scipy scikit-learn opencv-python-headless==4.11.0.86')
get_ipython().system('pip install nuscenes-devkit')
%matplotlib inline

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.2 MB/s eta 0:00:00
  Using cached opencv_python_headless-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (50.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.0/474.0 kB 29.6 MB/s eta 0:00:00
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.6.0
    Uninstalling threadpoolctl-3.6.0:
      Successfully uninstalled threadpoolctl-3.6.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
 

  Using cached numpy-1.26.4-cp313-cp313-linux_x86_64.whl
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 56.4 MB/s eta 0:00:00
^C


In [1]:
!ls /data/sets/nuscenes/v1.0-mini/

attribute.json		log.json		scene.json
calibrated_sensor.json	map.json		sensor.json
category.json		sample_annotation.json	visibility.json
ego_pose.json		sample_data.json
instance.json		sample.json


## A look at the dataset

### 1. `scene`

In [ ]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_DIR, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.813 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


nuScenes is a large scale database that features annotated samples across ***1000 scenes*** of approximately 20 seconds each. Let's take a look at the scenes that we have in the loaded database.

In [3]:
nusc.list_scenes()

scene-0061, Parked truck, construction, intersectio... [18-07-24 03:28:47]   19s, singapore-onenorth, #anns:4622
scene-0103, Many peds right, wait for turning car, ... [18-08-01 19:26:43]   19s, boston-seaport, #anns:2046
scene-0655, Parking lot, parked cars, jaywalker, be... [18-08-27 15:51:32]   20s, boston-seaport, #anns:2332
scene-0553, Wait at intersection, bicycle, large tr... [18-08-28 20:48:16]   20s, boston-seaport, #anns:1950
scene-0757, Arrive at busy intersection, bus, wait ... [18-08-30 19:25:08]   20s, boston-seaport, #anns:592
scene-0796, Scooter, peds on sidewalk, bus, cars, t... [18-10-02 02:52:24]   20s, singapore-queensto, #anns:708
scene-0916, Parking lot, bicycle rack, parked bicyc... [18-10-08 07:37:13]   20s, singapore-queensto, #anns:2387
scene-1077, Night, big street, bus stop, high speed... [18-11-21 11:39:27]   20s, singapore-hollandv, #anns:890
scene-1094, Night, after rain, many peds, PMD, ped ... [18-11-21 11:47:27]   19s, singapore-hollandv, #anns:1762
sc

Let's look at a scene metadata

In [4]:
my_scene = nusc.scene[0]
my_scene

{'token': 'cc8c0bf57f984915a77078b10eb33198',
 'log_token': '7e25a2c8ea1f41c5b0da1e69ecfa71a2',
 'nbr_samples': 39,
 'first_sample_token': 'ca9a282c9e77460f8360f564131a8af5',
 'last_sample_token': 'ed5fc18c31904f96a8f0dbb99ff069c0',
 'name': 'scene-0061',
 'description': 'Parked truck, construction, intersection, turn left, following a van'}

### 2. `sample`

In scenes, we annotate our data every half a second (2 Hz).

We define `sample` as an ***annotated keyframe of a scene at a given timestamp***. A keyframe is a frame where the time-stamps of data from all the sensors should be very close to the time-stamp of the sample it points to.

Now, let us look at the first annotated sample in this scene.

In [5]:
first_sample_token = my_scene['first_sample_token']

# The rendering command below is commented out because it tends to crash in notebooks
# nusc.render_sample(first_sample_token)

Let's examine its metadata

In [6]:
my_sample = nusc.get('sample', first_sample_token)
my_sample

{'token': 'ca9a282c9e77460f8360f564131a8af5',
 'timestamp': 1532402927647951,
 'prev': '',
 'next': '39586f9d59004284a7114a68825e8eec',
 'scene_token': 'cc8c0bf57f984915a77078b10eb33198',
 'data': {'RADAR_FRONT': '37091c75b9704e0daa829ba56dfa0906',
  'RADAR_FRONT_LEFT': '11946c1461d14016a322916157da3c7d',
  'RADAR_FRONT_RIGHT': '491209956ee3435a9ec173dad3aaf58b',
  'RADAR_BACK_LEFT': '312aa38d0e3e4f01b3124c523e6f9776',
  'RADAR_BACK_RIGHT': '07b30d5eb6104e79be58eadf94382bc1',
  'LIDAR_TOP': '9d9bf11fb0e144c8b446d54a8a00184f',
  'CAM_FRONT': 'e3d495d4ac534d54b321f50006683844',
  'CAM_FRONT_RIGHT': 'aac7867ebf4f446395d29fbd60b63b3b',
  'CAM_BACK_RIGHT': '79dbb4460a6b40f49f9c150cb118247e',
  'CAM_BACK': '03bea5763f0f4722933508d5999c5fd8',
  'CAM_BACK_LEFT': '43893a033f9c46d4a51b5e08a67a1eb7',
  'CAM_FRONT_LEFT': 'fe5422747a7d4268a4b07fc396707b23'},
 'anns': ['ef63a697930c4b20a6b9791f423351da',
  '6b89da9bf1f84fd6a5fbe1c3b236f809',
  '924ee6ac1fed440a9d9e3720aac635a0',
  '91e3608f55174a319

A useful method is  `list_sample()` which lists all related `sample_data` keyframes and `sample_annotation` associated with a `sample` which we will discuss in detail in the subsequent parts.

In [7]:
nusc.list_sample(my_sample['token'])

Sample: ca9a282c9e77460f8360f564131a8af5

sample_data_token: 37091c75b9704e0daa829ba56dfa0906, mod: radar, channel: RADAR_FRONT
sample_data_token: 11946c1461d14016a322916157da3c7d, mod: radar, channel: RADAR_FRONT_LEFT
sample_data_token: 491209956ee3435a9ec173dad3aaf58b, mod: radar, channel: RADAR_FRONT_RIGHT
sample_data_token: 312aa38d0e3e4f01b3124c523e6f9776, mod: radar, channel: RADAR_BACK_LEFT
sample_data_token: 07b30d5eb6104e79be58eadf94382bc1, mod: radar, channel: RADAR_BACK_RIGHT
sample_data_token: 9d9bf11fb0e144c8b446d54a8a00184f, mod: lidar, channel: LIDAR_TOP
sample_data_token: e3d495d4ac534d54b321f50006683844, mod: camera, channel: CAM_FRONT
sample_data_token: aac7867ebf4f446395d29fbd60b63b3b, mod: camera, channel: CAM_FRONT_RIGHT
sample_data_token: 79dbb4460a6b40f49f9c150cb118247e, mod: camera, channel: CAM_BACK_RIGHT
sample_data_token: 03bea5763f0f4722933508d5999c5fd8, mod: camera, channel: CAM_BACK
sample_data_token: 43893a033f9c46d4a51b5e08a67a1eb7, mod: camera, channel:

### 3. `sample_data`

The nuScenes dataset contains data that is collected from a full sensor suite. Hence, for each snapshot of a scene, we provide references to a family of data that is collected from these sensors.

We provide a `data` key to access these:

In [8]:
my_sample['data']

{'RADAR_FRONT': '37091c75b9704e0daa829ba56dfa0906',
 'RADAR_FRONT_LEFT': '11946c1461d14016a322916157da3c7d',
 'RADAR_FRONT_RIGHT': '491209956ee3435a9ec173dad3aaf58b',
 'RADAR_BACK_LEFT': '312aa38d0e3e4f01b3124c523e6f9776',
 'RADAR_BACK_RIGHT': '07b30d5eb6104e79be58eadf94382bc1',
 'LIDAR_TOP': '9d9bf11fb0e144c8b446d54a8a00184f',
 'CAM_FRONT': 'e3d495d4ac534d54b321f50006683844',
 'CAM_FRONT_RIGHT': 'aac7867ebf4f446395d29fbd60b63b3b',
 'CAM_BACK_RIGHT': '79dbb4460a6b40f49f9c150cb118247e',
 'CAM_BACK': '03bea5763f0f4722933508d5999c5fd8',
 'CAM_BACK_LEFT': '43893a033f9c46d4a51b5e08a67a1eb7',
 'CAM_FRONT_LEFT': 'fe5422747a7d4268a4b07fc396707b23'}

Notice that the keys are referring to the different sensors that form our sensor suite. Let's take a look at the metadata of a `sample_data` taken from `CAM_FRONT`.

In [25]:
sensor = 'LIDAR_TOP'
lidar_top_data = nusc.get('sample_data', my_sample['data'][sensor])
lidar_top_data

{'token': '9d9bf11fb0e144c8b446d54a8a00184f',
 'sample_token': 'ca9a282c9e77460f8360f564131a8af5',
 'ego_pose_token': '9d9bf11fb0e144c8b446d54a8a00184f',
 'calibrated_sensor_token': 'a183049901c24361a6b0b11b8013137c',
 'timestamp': 1532402927647951,
 'fileformat': 'pcd',
 'is_key_frame': True,
 'height': 0,
 'width': 0,
 'filename': 'samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402927647951.pcd.bin',
 'prev': '',
 'next': '0cedf1d2d652468d92d23491136b5d15',
 'sensor_modality': 'lidar',
 'channel': 'LIDAR_TOP'}

In [26]:
calibration_data = nusc.get('calibrated_sensor', lidar_top_data['calibrated_sensor_token'])
calibration_data

{'token': 'a183049901c24361a6b0b11b8013137c',
 'sensor_token': 'dc8b396651c05aedbb9cdaae573bb567',
 'translation': [0.943713, 0.0, 1.84023],
 'rotation': [0.7077955119163518,
  -0.006492242056004365,
  0.010646214713995808,
  -0.7063073142877817],
 'camera_intrinsic': []}

### Extract all sample tokens from a scene

In [50]:
def get_sample_tokens_from_scene(scene_token):
    # get the scene dict using the scene_token
    scene = nusc.get('scene', scene_token)

    # Extract the first and last sample token for the scene
    sample_token = scene['first_sample_token']
    last_sample_token = scene['last_sample_token']

    # Loop over the sample tokens belonging to a scene by using the
    # 'next' attribute in the sample dict
    sample_tokens = [sample_token]
    while sample_token != last_sample_token:
        sample = nusc.get('sample', sample_token)
        next_sample_token = sample['next']
        sample_tokens.append(next_sample_token)
        sample_token = next_sample_token

    sample_tokens.append(last_sample_token)

    return sample_tokens

### Retrieve sample data with channel and calibration matrix

In [ ]:
def extract_sensor_data_from_sample_data(sample_data_token):
    # get sample data record using sample data token
    sd_record = nusc.get('sample_data', sample_data_token)

    sensor_data = {}

    # Get sensor modality and channel
    sensor_modality = sd_record['sensor_modality']
    channel = sd_record['channel']

    # Get ego pose token
    ego_pose_token = sd_record['ego_pose_token']
    ego_pose = nusc.get('ego_pose', ego_pose_token)
    sensor_data["ego_pose"] = {"translation": ego_pose['translation'], "rotation": ego_pose['rotation'], "timestamp": ego_pose['timestamp']}

    # Extract sample metadata
    sensor_data["timestamp"] = sd_record['timestamp']
    sensor_data["filepath"] = sd_record['filename']
    sensor_data["fileformat"] = sd_record['fileformat']
    sensor_data["image_height"] = sd_record['height']
    sensor_data["image_width"] = sd_record['width']

    # Retrieve calibration matrix
    calibrated_sensor_token = sd_record['calibrated_sensor_token']
    calibrated_sensor_record = nusc.get('calibrated_sensor', calibrated_sensor_token)
    sensor_data["sensor_intrinsic"] = calibrated_sensor_record['camera_intrinsic']

    # Retrive extrinsic parameters
    sensor_data["sensor_extrinsic"] = {"translation": calibrated_sensor_record['translation'], "rotation": calibrated_sensor_record['rotation']}

    return sensor_modality, channel, sensor_data


In [12]:

import math

# Get skew of sample data timestamp from sample timestamp
def calculate_timestamp_skew_ms(sample_ts, sample_data_ts):
    # sample_ts and sample_data_ts are in microseconds (UNIX timestamp)
    # dividing by 1000 gives the difference in milliseconds
    return math.fabs(sample_data_ts - sample_ts) / 1000


In [ ]:
def extract_sample_data_from_sample(sample_token):
    # get sample data tokens per channel
    sample = nusc.get('sample', sample_token)
    sample_data_tokens = sample['data']
    sample_ts = sample['timestamp']

    # Initialize the sample aggregate dict
    sample_data = {"sample_token": sample_token, "scene_token": sample['scene_token'],
                   "reference_timestamp": sample_ts}
    # Variable to track the maximum skew of a smaple data from the reference timestamp
    max_camera_skew_ms = 0

    for channel, data_token in sample_data_tokens.items():
        # extract sensor data
        sensor_modality, channel, sensor_data = extract_sensor_data_from_sample_data(data_token)

        # get timestamp skew
        sample_data_ts = sensor_data["timestamp"]

        # Calculate the skew (absolute difference) between the sample timestamp and the sample data timestamp
        # The UNIX timestamps are recorded in microseconds and the skew is calculated in milliseconds
        timestamp_skew_ms = calculate_timestamp_skew_ms(sample_ts, sample_data_ts)
        sensor_data["ts_skew_ms"] = timestamp_skew_ms
        max_camera_skew_ms = max(max_camera_skew_ms, timestamp_skew_ms)

        # Add sensor data to sample aggregate dict
        # Keyed by sensor modality and channel
        if sensor_modality in sample_data:
            sample_data[sensor_modality][channel] = sensor_data
        else:
            sample_data[sensor_modality] = {channel: sensor_data}

    # Add max skew to the aggregate dict
    sample_data["max_camera_skew_ms"] = max_camera_skew_ms

    return sample_data


In [ ]:
# Write the extracted data to a output JSON file
import json
import os

# Create the outut directory to store the aggregated data files
output_dir = "output/nuscenes/v1/scenes"
os.makedirs(output_dir, exist_ok=True)

# Loop over all scenes
for scene in nusc.scene:
    print("Processing scene {}".format(scene['name']))
    scene_token = scene['token']
    sample_tokens = get_sample_tokens_from_scene(scene_token)
    scene_data = {}

    # Loop over each sample and extract corresponding sample data along with sensor calibration and time skew
    for sample_token in sample_tokens:
        print("Processing sample {}".format(sample_token))
        sample_data = extract_sample_data_from_sample(sample_token)
        scene_data[sample_token] = sample_data


    # Write to a aggregate JSON file corresponding to each scene
    output_path = os.path.join(output_dir, scene_token + ".json")
    with open(output_path, 'w') as f:
        json.dump(scene_data, f)

Processing scene scene-0061
Processing sample ca9a282c9e77460f8360f564131a8af5
Processing sample 39586f9d59004284a7114a68825e8eec
Processing sample 356d81f38dd9473ba590f39e266f54e5
Processing sample e0845f5322254dafadbbed75aaa07969
Processing sample c923fe08b2ff4e27975d2bf30934383b
Processing sample f1e3d9d08f044c439ce86a2d6fcca57b
Processing sample 4f545737bf3347fbbc9af60b0be9a963
Processing sample 7626dde27d604ac28a0240bdd54eba7a
Processing sample be99ffc878b24aca8956bbb4e0f97d0c
Processing sample 9813c23a5f1448b09bb7910fea9baf20
Processing sample 023c4df2d451409881d8e6ea82f14704
Processing sample c235638ed66145988d17f9d0601923f2
Processing sample bc3c8a953f6b4dcdb77b521d89f3d9d5
Processing sample 1e3d79dae62742a0ad64c91679863358
Processing sample 2afb9d32310e4546a71cbe432911eca2
Processing sample cd21dbfc3bd749c7b10a5c42562e0c42
Processing sample 88449a5cb1644a199c1c11f6ac034867
Processing sample 2ff86dc19c4644a1a88ce5ba848f56e5
Processing sample bf2938e43c6f487497cda76b51bfc406
Pro

### Plot of max camera skew
Camera skew: the time difference between when a given camera captured its frame and a common reference timestamp (timestamp of the sample).

It's a per-camera quantity. Each of the 6 cameras, 5 radar sensors and 1 lidar sensor gets its own skew value for a given sample, since each camera's shutter fires independently and lands at a slightly different real time than the others, even though they're all being grouped together as "the same moment."

`max_camera_skew_ms` is the largest of the six individual skew values in a sample, used for the threshold/flagging decision, since a bundle is only as trustworthy as its worst-aligned camera.

NOTE: We add a threshold to visualize which samples should be flagged for excessive skew. This is displayed as a line in the scatter plot.

In [38]:
SKEW_THRESHOLD = 43.5

In [ ]:
import numpy as np

skews_arr = np.array([sd["max_camera_skew_ms"] for _, sd in scene_data.items()])
sample_tokens = list(scene_data.keys())

### Histogram

In [ ]:
import plotly.graph_objects as go

# --- Plain histogram, no threshold line ---
bin_edges = np.histogram_bin_edges(skews_arr, bins="fd")
bin_width = bin_edges[1] - bin_edges[0]

fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=skews_arr,
    xbins=dict(start=bin_edges[0], end=bin_edges[-1], size=bin_width),
))
fig_hist.update_xaxes(range=[0, SKEW_THRESHOLD * 1.05], autorange=False)
fig_hist.update_layout(
    title=f"Max camera skew per sample (bin width={bin_width:.1f}ms, Freedman-Diaconis)",
    xaxis_title="Max camera skew (ms)",
    yaxis_title="Count",
)
fig_hist.show()

### Scatter Plot with threshold line

In [ ]:
import pandas as pd
import plotly.express as px

# --- Scatter, with threshold line + shaded discard region ---
df = pd.DataFrame({
    "sample_index": range(1, len(skews_arr) + 1),
    "skew_ms": skews_arr,
    "sample_token": sample_tokens,
})
df["exceeds_threshold"] = df["skew_ms"] > SKEW_THRESHOLD

fig_scatter = px.scatter(
    df, x="sample_index", y="skew_ms",
    color="exceeds_threshold",
    color_discrete_map={True: "crimson", False: "steelblue"},
    hover_data=["sample_token"],
)

fig_scatter.add_hrect(
    y0=SKEW_THRESHOLD, y1=skews_arr.max() * 1.05,
    fillcolor="red", opacity=0.08, layer="below", line_width=0,
)
fig_scatter.add_hline(
    y=SKEW_THRESHOLD, line_dash="dash", line_color="red",
    annotation_text=f"{SKEW_THRESHOLD}ms threshold", annotation_position="top right",
)

pct_flagged = df["exceeds_threshold"].mean() * 100
fig_scatter.update_xaxes(range=[0, SKEW_THRESHOLD * 1.05], autorange=False)
fig_scatter.update_layout(
    title=f"Max camera skew by sample ({pct_flagged:.1f}% flagged for discard)",
    xaxis_title="Sample index",
    yaxis_title="Max camera skew (ms)",
    legend_title="Exceeds threshold",
)
fig_scatter.show()

## Plot of per-camera skew

Per-camera skew distribution is the individual, non-maxed skew values, used in the box plot to see whether misalignment is spread evenly across cameras or concentrated in one.

### Box plot of per-camera skew

In [ ]:
import plotly.express as px
import pandas as pd

rows = []
for _, sd in scene_data.items():
    # Add any cameras
    if "camera" in sd:
        for cam, data in sd["camera"].items():
            rows.append({"sensor": cam, "skew_ms": data["ts_skew_ms"], "sample_token": sd["sample_token"]})

    # Add any radar sensors
    if "radar" in sd:
        for cam, data in sd["radar"].items():
            rows.append({"sensor": cam, "skew_ms": data["ts_skew_ms"], "sample_token": sd["sample_token"]})

    # Add any lidar sensors
    if "lidar" in sd:
        for cam, data in sd["lidar"].items():
            rows.append({"sensor": cam, "skew_ms": data["ts_skew_ms"], "sample_token": sd["sample_token"]})

df = pd.DataFrame(rows)

fig = px.box(df, x="sensor", y="skew_ms", points="outliers", hover_data=["sample_token"])
fig.add_hline(y=SKEW_THRESHOLD, line_dash="dash", line_color="red")
fig.update_yaxes(rangemode="tozero")
fig.update_layout(title="Per-camera skew distribution", xaxis_title="Camera", yaxis_title="Max camera skew (ms)")
fig.show()

['CAM_FRONT_LEFT' 'RADAR_BACK_RIGHT' 'LIDAR_TOP']


### 4. `sample_annotation`

We can also render the `sample_data` at a particular sensor.

In [ ]:
nusc.render_sample_data(lidar_top_data['token'])

`sample_annotation` refers to any ***bounding box defining the position of an object seen in a sample***. All location data is given with respect to the global coordinate system. Let's examine an example from our `sample` above.

In [ ]:
my_annotation_token = my_sample['anns'][18]
my_annotation_metadata =  nusc.get('sample_annotation', my_annotation_token)
my_annotation_metadata

We can also render an annotation to have a closer look.

In [ ]:
nusc.render_annotation(my_annotation_token)

### 5. `instance`

Object instance are instances that need to be detected or tracked by an AV (e.g a particular vehicle, pedestrian). Let us examine an instance metadata

In [ ]:
my_instance = nusc.instance[599]
my_instance

We generally track an instance across different frames in a particular scene. However, we do not track them across different scenes. In this example, we have 16 annotated samples for this instance across a particular scene.

In [ ]:
instance_token = my_instance['token']
nusc.render_instance(instance_token)

An instance record takes note of its first and last annotation token. Let's render them

In [ ]:
print("First annotated sample of this instance:")
nusc.render_annotation(my_instance['first_annotation_token'])

In [ ]:
print("Last annotated sample of this instance")
nusc.render_annotation(my_instance['last_annotation_token'])

### 6. `category`

A `category` is the object assignment of an annotation.  Let's look at the category table we have in our database. The table contains the taxonomy of different object categories and also list the subcategories (delineated by a period).

In [ ]:
nusc.list_categories()

A category record contains the name and the description of that particular category.

In [ ]:
nusc.category[9]

Refer to `instructions_nuscenes.md` for the definitions of the different categories.

### 7. `attribute`

An `attribute` is a property of an instance that may change throughout different parts of a scene while the category remains the same. Here we list the provided attributes and the number of annotations associated with a particular attribute.

In [ ]:
nusc.list_attributes()

Let's take a look at an example how an attribute may change over one scene

In [ ]:
my_instance = nusc.instance[27]
first_token = my_instance['first_annotation_token']
last_token = my_instance['last_annotation_token']
nbr_samples = my_instance['nbr_annotations']
current_token = first_token

i = 0
found_change = False
while current_token != last_token:
    current_ann = nusc.get('sample_annotation', current_token)
    current_attr = nusc.get('attribute', current_ann['attribute_tokens'][0])['name']

    if i == 0:
        pass
    elif current_attr != last_attr:
        print("Changed from `{}` to `{}` at timestamp {} out of {} annotated timestamps".format(last_attr, current_attr, i, nbr_samples))
        found_change = True

    next_token = current_ann['next']
    current_token = next_token
    last_attr = current_attr
    i += 1

### 8. `visibility`

`visibility` is defined as the fraction of pixels of a particular annotation that are visible over the 6 camera feeds, grouped into 4 bins.

In [ ]:
nusc.visibility

Let's look at an example `sample_annotation` with 80-100% visibility

In [ ]:
anntoken = 'a7d0722bce164f88adf03ada491ea0ba'
visibility_token = nusc.get('sample_annotation', anntoken)['visibility_token']

print("Visibility: {}".format(nusc.get('visibility', visibility_token)))
nusc.render_annotation(anntoken)

Let's look at an example `sample_annotation` with 0-40% visibility

In [ ]:
anntoken = '9f450bf6b7454551bbbc9a4c6e74ef2e'
visibility_token = nusc.get('sample_annotation', anntoken)['visibility_token']

print("Visibility: {}".format(nusc.get('visibility', visibility_token)))
nusc.render_annotation(anntoken)

### 9. `sensor`

The nuScenes dataset consists of data collected from our full sensor suite which consists of:
- 1 x LIDAR,
- 5 x RADAR,
- 6 x cameras,

In [ ]:
nusc.sensor

Every `sample_data` has a record on which `sensor` the data is collected from (note the "channel" key)

In [ ]:
nusc.sample_data[10]

### 10. `calibrated_sensor`

`calibrated_sensor` consists of the definition of a particular sensor (lidar/radar/camera) as calibrated on a particular vehicle. Let us look at an example.

In [ ]:
nusc.calibrated_sensor[0]

Note that the `translation` and the `rotation` parameters are given with respect to the ego vehicle body frame.

### 11. `ego_pose`

`ego_pose` contains information about the location (encoded in `translation`) and the orientation (encoded in `rotation`) of the ego vehicle, with respect to the global coordinate system.

In [ ]:
nusc.ego_pose[0]

Note that the number of `ego_pose` records in our loaded database is the same as the number of `sample_data` records. These two records exhibit a one-to-one correspondence.

### 12. `log`

The `log` table contains log information from which the data was extracted. A `log` record corresponds to one journey of our ego vehicle along a predefined route. Let's check the number of logs and the metadata of a log.

In [ ]:
print("Number of `logs` in our loaded database: {}".format(len(nusc.log)))

In [ ]:
nusc.log[0]

Notice that it contains a variety of information such as the date and location of the log collected. It also gives out information about the map from where the data was collected. Note that one log can contain multiple non-overlapping scenes.

### 13. `map`

Map information is stored as binary semantic masks from a top-down view. Let's check the number of maps and metadata of a map.

In [ ]:
print("There are {} maps masks in the loaded dataset".format(len(nusc.map)))

In [ ]:
nusc.map[0]

## nuScenes Basics

Let's get a bit technical.

The NuScenes class holds several tables. Each table is a list of records, and each record is a dictionary. For example the first record of the category table is stored at:

In [ ]:
nusc.category[0]

The category table is simple: it holds the fields `name` and `description`. It also has a `token` field, which is a unique record identifier. Since the record is a dictionary, the token can be accessed like so:

In [ ]:
cat_token = nusc.category[0]['token']
cat_token

If you know the `token` for any record in the DB you can retrieve the record by doing

In [ ]:
nusc.get('category', cat_token)

_As you can notice, we have recovered the same record!_

OK, that was easy. Let's try something harder. Let's look at the `sample_annotation` table.

In [ ]:
nusc.sample_annotation[0]

This also has a `token` field (they all do). In addition, it has several fields of the format [a-z]*\_token, _e.g._ instance_token. These are foreign keys in database terminology, meaning they point to another table.
Using `nusc.get()` we can grab any of these in constant time. For example, let's look at the visibility record.

In [ ]:
nusc.get('visibility', nusc.sample_annotation[0]['visibility_token'])

The visibility records indicate how much of an object was visible when it was annotated.

Let's also grab the `instance_token`

In [ ]:
one_instance = nusc.get('instance', nusc.sample_annotation[0]['instance_token'])
one_instance

This points to the `instance` table. This table enumerate the object _instances_ we have encountered in each
scene. This way we can connect all annotations of a particular object.

If you look carefully at the README tables, you will see that the sample_annotation table points to the instance table,
but the instance table doesn't list all annotations that point to it.

So how can we recover all sample_annotations for a particular object instance? There are two ways:

1. `Use nusc.field2token()`. Let's try it:

In [ ]:
ann_tokens = nusc.field2token('sample_annotation', 'instance_token', one_instance['token'])

This returns a list of all sample_annotation records with the `'instance_token'` == `one_instance['token']`. Let's store these in a set for now

In [ ]:
ann_tokens_field2token = set(ann_tokens)

ann_tokens_field2token

The `nusc.field2token()` method is generic and can be used in any similar situation.

2. For certain situation, we provide some reverse indices in the tables themselves. This is one such example.

The instance record has a field `first_annotation_token` which points to the first annotation in time of this instance.
Recovering this record is easy.

In [ ]:
ann_record = nusc.get('sample_annotation', one_instance['first_annotation_token'])
ann_record

Now we can traverse all annotations of this instance using the "next" field. Let's try it.

In [ ]:
ann_tokens_traverse = set()
ann_tokens_traverse.add(ann_record['token'])
while not ann_record['next'] == "":
    ann_record = nusc.get('sample_annotation', ann_record['next'])
    ann_tokens_traverse.add(ann_record['token'])

Finally, let's assert that we recovered the same ann_records as we did using nusc.field2token:

In [ ]:
print(ann_tokens_traverse == ann_tokens_field2token)

## Reverse indexing and short-cuts

The nuScenes tables are normalized, meaning that each piece of information is only given once.
For example, there is one `map` record for each `log` record. Looking at the schema you will notice that the `map` table has a `log_token` field, but that the `log` table does not have a corresponding `map_token` field. But there are plenty of situations where you have a `log`, and want to find the corresponding `map`! So what to do? You can always use the `nusc.field2token()` method, but that is slow and inconvenient. We therefore add reverse mappings for some common situations including this one.

Further, there are situations where one needs to go through several tables to get a certain piece of information.
Consider, for example, the category name (e.g. `human.pedestrian`) of a `sample_annotation`. The `sample_annotation` table doesn't hold this information since the category is an instance level constant. Instead the `sample_annotation` table points to a record in the `instance` table. This, in turn, points to a record in the `category` table, where finally the `name` fields stores the required information.

Since it is quite common to want to know the category name of an annotation, we add a `category_name` field to the `sample_annotation` table during initialization of the NuScenes class.

In this section, we list the short-cuts and reverse indices that are added to the `NuScenes` class during initialization. These are all created in the `NuScenes.__make_reverse_index__()` method.

### Reverse indices
We add two reverse indices by default.
* A `map_token` field is added to the `log` records.
* The `sample` records have shortcuts to all `sample_annotations` for that record as well as `sample_data` key-frames. Confer `nusc.list_sample()` method in the previous section for more details on this.

### Shortcuts

The sample_annotation table has a "category_name" shortcut.

_Using shortcut:_

In [ ]:
catname = nusc.sample_annotation[0]['category_name']

_Not using shortcut:_

In [ ]:
ann_rec = nusc.sample_annotation[0]
inst_rec = nusc.get('instance', ann_rec['instance_token'])
cat_rec = nusc.get('category', inst_rec['category_token'])

print(catname == cat_rec['name'])

The sample_data table has "channel" and "sensor_modality" shortcuts:

In [ ]:
# Shortcut
channel = nusc.sample_data[0]['channel']

# No shortcut
sd_rec = nusc.sample_data[0]
cs_record = nusc.get('calibrated_sensor', sd_rec['calibrated_sensor_token'])
sensor_record = nusc.get('sensor', cs_record['sensor_token'])

print(channel == sensor_record['channel'])

## Data Visualizations

We provide list and rendering methods. These are meant both as convenience methods during development, and as tutorials for building your own visualization methods. They are implemented in the NuScenesExplorer class, with shortcuts through the NuScenes class itself.

### List methods
There are three list methods available.

1. `list_categories()` lists all categories, counts and statistics of width/length/height in meters and aspect ratio.

In [ ]:
nusc.list_categories()

2. `list_attributes()` lists all attributes and counts.

In [ ]:
nusc.list_attributes()

3. `list_scenes()` lists all scenes in the loaded DB.

In [ ]:
nusc.list_scenes()

### Render

First, let's plot a lidar point cloud in an image. Lidar allows us to accurately map the surroundings in 3D.

In [ ]:
my_sample = nusc.sample[10]
nusc.render_pointcloud_in_image(my_sample['token'], pointsensor_channel='LIDAR_TOP')

In the previous image the colors indicate the distance from the ego vehicle to each lidar point. We can also render the lidar intensity. In the following image the traffic sign ahead of us is highly reflective (yellow) and the dark vehicle on the right has low reflectivity (purple).

In [ ]:
nusc.render_pointcloud_in_image(my_sample['token'], pointsensor_channel='LIDAR_TOP', render_intensity=True)

Second, let's plot the radar point cloud for the same image. Radar is less dense than lidar, but has a much larger range.

In [ ]:
nusc.render_pointcloud_in_image(my_sample['token'], pointsensor_channel='RADAR_FRONT')

We can also plot all annotations across all sample data for that sample. Note how for radar we also plot the velocity vectors of moving objects. Some velocity vectors are outliers, which can be filtered using the settings in RadarPointCloud.from_file()

In [ ]:
my_sample = nusc.sample[20]

# The rendering command below is commented out because it may crash in notebooks
# nusc.render_sample(my_sample['token'])

Or if we only want to render a particular sensor, we can specify that.

In [ ]:
nusc.render_sample_data(my_sample['data']['CAM_FRONT'])

Additionally we can aggregate the point clouds from multiple sweeps to get a denser point cloud.

In [ ]:
nusc.render_sample_data(my_sample['data']['LIDAR_TOP'], nsweeps=5, underlay_map=True)
nusc.render_sample_data(my_sample['data']['RADAR_FRONT'], nsweeps=5, underlay_map=True)

In the radar plot above we only see very confident radar returns from two vehicles. This is due to the filter settings defined in the file `nuscenes/utils/data_classes.py`. If instead we want to disable all filters and render all returns, we can use the `disable_filters()` function. This returns a denser point cloud, but with many returns from background objects. To return to the default settings, simply call `default_filters()`.

In [ ]:
from nuscenes.utils.data_classes import RadarPointCloud
RadarPointCloud.disable_filters()
nusc.render_sample_data(my_sample['data']['RADAR_FRONT'], nsweeps=5, underlay_map=True)
RadarPointCloud.default_filters()

We can even render a specific annotation.

In [ ]:
nusc.render_annotation(my_sample['anns'][22])

Finally, we can render a full scene as a video. There are two options here:
1. nusc.render_scene_channel() renders the video for a particular channel. (HIT ESC to exit)
2. nusc.render_scene() renders the video for all camera channels.

NOTE: These methods use OpenCV for rendering, which doesn't always play nice with IPython Notebooks. If you experience any issues please run these lines from the command line.

Let's grab scene 0061, it is nice and dense.

In [ ]:
my_scene_token = nusc.field2token('scene', 'name', 'scene-0061')[0]

In [ ]:
# The rendering command below is commented out because it may crash in notebooks
# nusc.render_scene_channel(my_scene_token, 'CAM_FRONT')

There is also a method nusc.render_scene() which renders the video for all camera channels.
This requires a high-res monitor, and is also best run outside this notebook.

In [ ]:
# The rendering command below is commented out because it may crash in notebooks
# nusc.render_scene(my_scene_token)

Finally, let us visualize all scenes on the map for a particular location.

In [ ]:
nusc.render_egoposes_on_map(log_location='singapore-onenorth')